# ICS 591 Team 2 — Stage 1: Model Selection for the Waste Verification Classifier

**Goal:** train five pretrained image classifiers under an *identical* protocol, compare them on metrics that reflect our system's real error costs, and select one model to carry forward into fine-tuning on local Madina data.

**Dataset:** RealWaste (UCI #908) — 4,752 images of real waste across 9 material classes, collected at a waste recovery facility. Used here as a **proxy task** (material classification) because our local clean/contaminated dataset does not exist yet.

**Candidates (all via `timm`, ImageNet-pretrained):**

| Model | Family | Why it's in the comparison |
|---|---|---|
| MobileNetV3-Large | Lightweight CNN | Cheapest to run; deployable on a low-cost server or phone |
| EfficientNet-B0 | Efficient CNN | Strong accuracy per parameter |
| ResNet-50 | Classic CNN | The standard baseline in waste-classification literature |
| ConvNeXt-Tiny | Modern CNN | Current-generation CNN design |
| ViT-S/16 | Vision Transformer | Transformer comparison at a similar parameter count |

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Expect roughly 30–50 minutes total. Results are saved after every model, so if Colab disconnects, just run all again and it resumes.

In [ ]:
# 1. Setup
!pip -q install timm
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# 2. Configuration — every model uses exactly these settings (fair comparison)
DATASET = 'realwaste'      # 'realwaste' (recommended) or 'trashnet' (backup)
SAVE_TO_DRIVE = True       # keeps results if Colab disconnects (you'll be asked to authorise)

MODELS = {
    'MobileNetV3-L':   'mobilenetv3_large_100.ra_in1k',
    'EfficientNet-B0': 'efficientnet_b0.ra_in1k',
    'ResNet-50':       'resnet50.a1_in1k',
    'ConvNeXt-Tiny':   'convnext_tiny.fb_in22k_ft_in1k',
    'ViT-S/16':        'vit_small_patch16_224.augreg_in21k_ft_in1k',
}

EPOCHS       = 12
BATCH_SIZE   = 32
LR           = 2e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTH = 0.1
IMG_SIZE     = 224
SEED         = 42
NUM_WORKERS  = 2
SPLIT        = (0.70, 0.15, 0.15)   # train / validation / test

TARGET_PLASTIC_RECALL = 0.95   # operating point for the accept/reject gate
REVIEW_CONF_THRESHOLD = 0.70   # below this confidence -> 'uncertain' -> manual review

In [ ]:
# 3. Output folder
import pathlib
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = pathlib.Path('/content/drive/MyDrive/ics591_model_selection') / DATASET
else:
    OUT = pathlib.Path('/content/outputs') / DATASET
OUT.mkdir(parents=True, exist_ok=True)
print('Saving results to', OUT)

In [ ]:
# 4. Download and locate the dataset
import zipfile, urllib.request

URLS = {
    'realwaste': 'https://archive.ics.uci.edu/static/public/908/realwaste.zip',
    'trashnet':  'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip',
}
RAW = pathlib.Path('/content/raw'); RAW.mkdir(parents=True, exist_ok=True)
zip_path = RAW / f'{DATASET}.zip'
extract_dir = RAW / DATASET

if not zip_path.exists():
    print('Downloading', URLS[DATASET], '...')
    urllib.request.urlretrieve(URLS[DATASET], zip_path)
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for _ in range(2):  # some archives contain zips inside the zip
        for inner in list(extract_dir.rglob('*.zip')):
            with zipfile.ZipFile(inner) as z:
                z.extractall(inner.parent)
            inner.unlink()

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def has_images(d):
    return any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file())

# find the folder whose sub-folders are the classes
best, best_n = None, 0
for d in [extract_dir, *[p for p in extract_dir.rglob('*') if p.is_dir()]]:
    if '__MACOSX' in d.parts:
        continue
    n = sum(1 for c in d.iterdir() if c.is_dir() and has_images(c))
    if n > best_n:
        best, best_n = d, n
assert best_n >= 2, 'Could not find class folders - inspect /content/raw in the file browser'
SRC_ROOT = best
CLASSES = sorted(c.name for c in SRC_ROOT.iterdir() if c.is_dir() and has_images(c))
print('Class folder root:', SRC_ROOT)
print(len(CLASSES), 'classes:', CLASSES)

In [ ]:
# 5. Pre-resize images once (short side 256px) so training isn't bottlenecked by decoding
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

DST_ROOT = pathlib.Path('/content/data') / DATASET
jobs = []
for ci, cname in enumerate(CLASSES):
    (DST_ROOT / cname).mkdir(parents=True, exist_ok=True)
    for f in sorted((SRC_ROOT / cname).iterdir()):
        if f.suffix.lower() in IMG_EXT:
            jobs.append((f, DST_ROOT / cname / (f.stem + '.jpg'), ci))

def resize_one(job):
    src, dst, ci = job
    if dst.exists():
        return (str(dst), ci)
    try:
        im = Image.open(src).convert('RGB')
        s = 256 / min(im.size)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
        im.save(dst, quality=95)
        return (str(dst), ci)
    except Exception as e:
        print('Skipping unreadable image', src, e)
        return None

with ThreadPoolExecutor(8) as ex:
    SAMPLES = [r for r in ex.map(resize_one, jobs) if r is not None]
print(len(SAMPLES), 'images ready')

In [ ]:
# 6. Stratified train / validation / test split (same split for every model)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

paths = np.array([p for p, _ in SAMPLES]); labels = np.array([y for _, y in SAMPLES])
tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, labels, test_size=1 - SPLIT[0],
                                            stratify=labels, random_state=SEED)
rel_test = SPLIT[2] / (SPLIT[1] + SPLIT[2])
va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=rel_test,
                                          stratify=tmp_y, random_state=SEED)
TRAIN, VAL, TEST = list(zip(tr_p, tr_y)), list(zip(va_p, va_y)), list(zip(te_p, te_y))

counts = pd.DataFrame({
    'train': np.bincount(tr_y, minlength=len(CLASSES)),
    'val':   np.bincount(va_y, minlength=len(CLASSES)),
    'test':  np.bincount(te_y, minlength=len(CLASSES)),
}, index=CLASSES)
counts.loc['TOTAL'] = counts.sum()
counts.to_csv(OUT / 'split_counts.csv')

PLASTIC = next(i for i, c in enumerate(CLASSES) if 'plastic' in c.lower())
print('"Accept" class for the recycling gate:', CLASSES[PLASTIC])
counts

In [ ]:
# 7. Data pipeline and training / evaluation helpers
import math, time, json, gc, random, copy
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class ImgDS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), int(y)

def make_tfms(model):
    cfg = timm.data.resolve_data_config({}, model=model)  # each model's own normalisation
    norm = T.Normalize(cfg['mean'], cfg['std'])
    train_tf = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.ColorJitter(0.3, 0.3, 0.3),   # simulates varied lighting and phone cameras
        T.ToTensor(), norm])
    eval_tf = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE), T.ToTensor(), norm])
    return train_tf, eval_tf

def loaders_for(model):
    train_tf, eval_tf = make_tfms(model)
    g = torch.Generator(); g.manual_seed(SEED)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    return (DataLoader(ImgDS(TRAIN, train_tf), BATCH_SIZE, shuffle=True, drop_last=True, generator=g, **kw),
            DataLoader(ImgDS(VAL, eval_tf), 64, **kw),
            DataLoader(ImgDS(TEST, eval_tf), 64, **kw))

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_one_epoch(model, loader, opt, sched, scaler, crit):
    model.train(); tot, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * x.size(0); n += x.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(DEVICE, non_blocking=True))
        P.append(logits.float().softmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

In [ ]:
# 8. Metrics tied to our system's error costs
def ece_score(probs, y, n_bins=15):
    # Expected Calibration Error: does a 0.8 confidence really mean 80% correct?
    conf, pred = probs.max(1), probs.argmax(1); correct = pred == y
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def plastic_gate(pv, yv, pt, yt):
    # Treat 'plastic' as ACCEPT and everything else as REJECT, like the real verifier.
    # Pick the threshold on VALIDATION so that plastic recall >= target, then measure on TEST.
    t = float(np.quantile(pv[yv == PLASTIC, PLASTIC], 1 - TARGET_PLASTIC_RECALL))
    accept = pt[:, PLASTIC] >= t; is_p = yt == PLASTIC
    return {'gate_threshold': t,
            'far_at_target': float(accept[~is_p].mean()),   # false accept: non-plastic paid out
            'frr_at_target': float(1 - accept[is_p].mean())}  # false reject: honest household turned away

def review_routing(pt, yt):
    conf = pt.max(1); auto = conf >= REVIEW_CONF_THRESHOLD
    return {'auto_decided_share': float(auto.mean()),
            'auto_decided_accuracy': float((pt.argmax(1)[auto] == yt[auto]).mean()) if auto.any() else float('nan')}

def latency_ms(model, device, runs):
    model = model.to(device).eval(); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(5): model(x)
        if device.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(runs): model(x)
        if device.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t) / runs * 1000

In [ ]:
# 9. Train and evaluate all five models (resumes automatically if interrupted)
RESULTS_FILE = OUT / 'results.json'
results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}

def slug(name): return name.replace('/', '_').replace(' ', '_')

for name, timm_id in MODELS.items():
    if name in results:
        print(f'[{name}] already done - skipping'); continue
    print(f'\n========== {name} ({timm_id}) ==========')
    set_seed(SEED)
    model = timm.create_model(timm_id, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
    train_dl, val_dl, test_dl = loaders_for(model)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total, warm = EPOCHS * len(train_dl), len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH); scaler = make_scaler()

    history, best_f1, best_state, best_ep = [], -1, None, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        loss = train_one_epoch(model, train_dl, opt, sched, scaler, crit)
        pv, yv = predict(model, val_dl)
        vf1 = f1_score(yv, pv.argmax(1), average='macro'); vacc = accuracy_score(yv, pv.argmax(1))
        history.append({'epoch': ep, 'train_loss': loss, 'val_macro_f1': vf1, 'val_acc': vacc})
        print(f'ep {ep:2d} | loss {loss:.3f} | val acc {vacc:.3f} | val macro-F1 {vf1:.3f} | {time.time()-t0:.0f}s')
        if vf1 > best_f1:   # model selection by validation, never by test
            best_f1, best_ep = vf1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    train_min = (time.time() - t_start) / 60

    model.load_state_dict(best_state)
    pv, yv = predict(model, val_dl); pt, yt = predict(model, test_dl)
    pred = pt.argmax(1)
    _, _, f1_per, _ = precision_recall_fscore_support(yt, pred, labels=range(len(CLASSES)), zero_division=0)

    ckpt = OUT / f'{slug(name)}.pt'; torch.save(best_state, ckpt)
    np.savez(OUT / f'{slug(name)}_test_probs.npz', probs=pt, y=yt)

    gpu_ms = latency_ms(model, DEVICE, 50) if DEVICE.type == 'cuda' else float('nan')
    cpu_ms = latency_ms(copy.deepcopy(model).float(), torch.device('cpu'), 20)

    results[name] = {
        'timm_id': timm_id, 'best_epoch': best_ep, 'train_minutes': train_min,
        'test_accuracy': float(accuracy_score(yt, pred)),
        'test_macro_f1': float(f1_score(yt, pred, average='macro')),
        'test_plastic_f1': float(f1_per[PLASTIC]),
        'per_class_f1': {c: float(f) for c, f in zip(CLASSES, f1_per)},
        'ece': ece_score(pt, yt),
        **plastic_gate(pv, yv, pt, yt),
        **review_routing(pt, yt),
        'params_m': sum(p.numel() for p in model.parameters()) / 1e6,
        'size_mb': ckpt.stat().st_size / 1e6,
        'gpu_latency_ms': gpu_ms, 'cpu_latency_ms': cpu_ms,
        'history': history,
    }
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"TEST: acc {results[name]['test_accuracy']:.3f} | macro-F1 {results[name]['test_macro_f1']:.3f} "
          f"| FAR@{TARGET_PLASTIC_RECALL:.0%} recall {results[name]['far_at_target']:.3f} "
          f"| ECE {results[name]['ece']:.3f} | CPU {cpu_ms:.0f} ms")
    del model, best_state, opt; gc.collect(); torch.cuda.empty_cache()

print('\nAll models done.')

In [ ]:
# 10. Comparison table
cols = ['test_accuracy', 'test_macro_f1', 'test_plastic_f1', 'far_at_target', 'frr_at_target', 'ece',
        'auto_decided_share', 'auto_decided_accuracy', 'params_m', 'size_mb',
        'cpu_latency_ms', 'gpu_latency_ms', 'train_minutes', 'best_epoch']
df = pd.DataFrame({n: {c: r[c] for c in cols} for n, r in results.items()}).T
df = df.sort_values('test_macro_f1', ascending=False)
df.to_csv(OUT / 'comparison_table.csv')
df.style.format('{:.3f}').background_gradient(subset=['test_macro_f1'], cmap='Greens') \
        .background_gradient(subset=['far_at_target', 'ece', 'cpu_latency_ms'], cmap='Reds')

In [ ]:
# 11. Are the differences real? Bootstrap 95% confidence intervals on the test set
rng = np.random.default_rng(SEED); B = 1000; rows = {}
for n in results:
    d = np.load(OUT / f'{slug(n)}_test_probs.npz'); pred, y = d['probs'].argmax(1), d['y']
    scores = [f1_score(y[i], pred[i], average='macro')
              for i in (rng.integers(0, len(y), len(y)) for _ in range(B))]
    rows[n] = {'macro_f1': f1_score(y, pred, average='macro'),
               'ci_low': np.percentile(scores, 2.5), 'ci_high': np.percentile(scores, 97.5)}
ci = pd.DataFrame(rows).T.sort_values('macro_f1', ascending=False)
ci.to_csv(OUT / 'macro_f1_bootstrap_ci.csv')
print('If two models\' intervals overlap heavily, accuracy alone cannot separate them -')
print('that is when calibration, false-accept rate, latency and size decide.')
ci.round(3)

In [ ]:
# 12. Charts for the report
import matplotlib.pyplot as plt
names = list(df.index)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].bar(names, ci.loc[names, 'macro_f1'],
          yerr=[ci.loc[names, 'macro_f1'] - ci.loc[names, 'ci_low'], ci.loc[names, 'ci_high'] - ci.loc[names, 'macro_f1']],
          capsize=4, color='#4C8BF5')
ax[0].set_title('Test macro-F1 (95% bootstrap CI)'); ax[0].tick_params(axis='x', rotation=30)
ax[0].set_ylim(max(0, ci['ci_low'].min() - 0.05), 1)

ax[1].bar(names, df['far_at_target'], color='#E8584A')
ax[1].set_title(f'False-accept rate at {TARGET_PLASTIC_RECALL:.0%} plastic recall (lower is better)')
ax[1].tick_params(axis='x', rotation=30)

for n in names:
    ax[2].scatter(df.loc[n, 'cpu_latency_ms'], df.loc[n, 'test_macro_f1'], s=df.loc[n, 'params_m'] * 15, alpha=.6)
    ax[2].annotate(n, (df.loc[n, 'cpu_latency_ms'], df.loc[n, 'test_macro_f1']), fontsize=9)
ax[2].set_xlabel('CPU latency per image (ms)'); ax[2].set_ylabel('Test macro-F1')
ax[2].set_title('Accuracy vs. speed (bubble size = parameters)')
plt.tight_layout(); plt.savefig(OUT / 'comparison_charts.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for n in names:
    h = pd.DataFrame(results[n]['history'])
    ax[0].plot(h['epoch'], h['val_macro_f1'], marker='o', label=n)
    ax[1].plot(h['epoch'], h['train_loss'], marker='o', label=n)
ax[0].set_title('Validation macro-F1 per epoch'); ax[1].set_title('Training loss per epoch')
for a in ax: a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.savefig(OUT / 'training_curves.png', dpi=150); plt.show()

In [ ]:
# 13. Calibration (reliability diagram) — matters because 'uncertain' routing uses the confidence score
plt.figure(figsize=(7, 6)); plt.plot([0, 1], [0, 1], 'k--', label='perfect calibration')
edges = np.linspace(0, 1, 11)
for n in names:
    d = np.load(OUT / f'{slug(n)}_test_probs.npz'); p, y = d['probs'], d['y']
    conf, corr = p.max(1), p.argmax(1) == y; xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5: xs.append(conf[m].mean()); ys.append(corr[m].mean())
    plt.plot(xs, ys, marker='o', label=f"{n} (ECE {results[n]['ece']:.3f})")
plt.xlabel('Model confidence'); plt.ylabel('Actual accuracy'); plt.title('Reliability diagram (test set)')
plt.legend(); plt.savefig(OUT / 'reliability_diagram.png', dpi=150); plt.show()

In [ ]:
# 14. Confusion matrices for the top two models
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(1, 2, figsize=(18, 7))
for a, n in zip(ax, names[:2]):
    d = np.load(OUT / f'{slug(n)}_test_probs.npz')
    ConfusionMatrixDisplay.from_predictions(d['y'], d['probs'].argmax(1), display_labels=CLASSES,
        normalize='true', values_format='.2f', ax=a, colorbar=False, xticks_rotation=45)
    a.set_title(n)
plt.tight_layout(); plt.savefig(OUT / 'confusion_matrices.png', dpi=150); plt.show()

pd.DataFrame({n: results[n]['per_class_f1'] for n in names}).round(3)

In [ ]:
# 15. Weighted selection score
# These weights are a DESIGN DECISION — justify them in the report from our error costs:
#  - macro-F1: overall ability to tell waste types apart
#  - false-accept rate: every false accept is a fraudulent payout or contaminated material sent to a recycler
#  - ECE: the 'uncertain -> manual review' path only works if confidence is trustworthy
#  - CPU latency and size: riders wait at pickup; we cannot assume a GPU server
WEIGHTS = {'test_macro_f1': 0.35, 'far_at_target': 0.25, 'ece': 0.15, 'cpu_latency_ms': 0.15, 'size_mb': 0.10}
HIGHER_IS_BETTER = {'test_macro_f1'}

score = pd.DataFrame(index=df.index)
for m, w in WEIGHTS.items():
    v = df[m].astype(float); rng_ = v.max() - v.min()
    norm = (v - v.min()) / rng_ if rng_ > 0 else v * 0 + 1
    score[m] = (norm if m in HIGHER_IS_BETTER else 1 - norm) * w
score['TOTAL'] = score.sum(1)
score = score.sort_values('TOTAL', ascending=False)
score.to_csv(OUT / 'selection_scores.csv')

WINNER = score.index[0]
print(f'Selected model: {WINNER}  ({results[WINNER]["timm_id"]})')
print(f'Checkpoint for Stage 2 fine-tuning: {OUT / (slug(WINNER) + ".pt")}')
score.round(3)

## Writing up the selection (for the report)

Use the saved files in your Drive folder (`comparison_table.csv`, `macro_f1_bootstrap_ci.csv`, `selection_scores.csv` and the PNG charts). A strong justification covers:

1. **Protocol** — same data split, image size, augmentation, optimiser, learning rate, epochs and seed for all five; best epoch chosen on validation; test set used once.
2. **Headline comparison** — the comparison table and the macro-F1 chart with confidence intervals. Say plainly whether the top models are statistically separable.
3. **Cost-aware metrics** — why false-accept rate at a fixed plastic recall and calibration matter more for our system than raw accuracy (link to the failure-mode table in C2 and the monitoring signals in C4).
4. **Deployment constraints** — CPU latency and model size, given mixed connectivity and no guaranteed GPU.
5. **Decision** — the weighted score, the weights and why, and why the winner beat each of the other four.
6. **Limitations** — RealWaste is Australian landfill waste labelled by *material*, not clean vs. contaminated. The ranking must be re-checked on local Madina photos before fine-tuning.